Importing Libraries

In [107]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

Importing the Dataset : https://raw.githubusercontent.com/omairaasim/machine_learning/master/project_11_k_nearest_neighbor/iphone_purchase_records.csv

In [108]:
df=pd.read_csv('https://raw.githubusercontent.com/omairaasim/machine_learning/master/project_11_k_nearest_neighbor/iphone_purchase_records.csv')

In [109]:
df.head()

,Gender,Age,Salary,Purchase Iphone
0,Male,19,19000,0
1,Male,35,20000,0
2,Female,26,43000,0
3,Female,27,57000,0
4,Male,19,76000,0


In [110]:
df.rename(columns={'Purchase Iphone':'Purchase'},inplace=True)

In [111]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 400 entries, 0 to 399
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   Gender    400 non-null    object
 1   Age       400 non-null    int64 
 2   Salary    400 non-null    int64 
 3   Purchase  400 non-null    int64 
dtypes: int64(3), object(1)
memory usage: 12.6+ KB


In [112]:
df.isnull().sum()

Gender      0
Age         0
Salary      0
Purchase    0
dtype: int64

In [113]:
df.describe()

,Age,Salary,Purchase
count,400.000000,400.000000,400.000000
mean,37.655000,69742.500000,0.357500
std,10.482877,34096.960282,0.479864
min,18.000000,15000.000000,0.000000
25%,29.750000,43000.000000,0.000000
50%,37.000000,70000.000000,0.000000
75%,46.000000,88000.000000,1.000000
max,60.000000,150000.000000,1.000000


EDA

In [114]:
df.Gender.value_counts()

Gender
Female    204
Male      196
Name: count, dtype: int64

In [115]:
df.loc[df['Purchase']==1,'Gender'].value_counts()

Gender
Female    77
Male      66
Name: count, dtype: int64

In [116]:
df.head()

,Gender,Age,Salary,Purchase
0,Male,19,19000,0
1,Male,35,20000,0
2,Female,26,43000,0
3,Female,27,57000,0
4,Male,19,76000,0


Splitting the data

In [117]:
X=df.iloc[:,:-1]
Y=df.iloc[:,-1]

In [118]:
X.head()

,Gender,Age,Salary
0,Male,19,19000
1,Male,35,20000
2,Female,26,43000
3,Female,27,57000
4,Male,19,76000


Encoding

In [119]:
X=pd.get_dummies(X,dtype=int,drop_first=True)

In [120]:
X.head()

,Age,Salary,Gender_Male
0,19,19000,1
1,35,20000,1
2,26,43000,0
3,27,57000,0
4,19,76000,1


Cross Validation Split

In [121]:
X.shape

(400, 3)

In [122]:
skf=StratifiedKFold(n_splits=5)

In [123]:
for train_index,test_index in skf.split(X,Y):
    X_train,X_test=X.iloc[train_index],X.iloc[test_index]
    Y_train,Y_test=Y.iloc[train_index],Y.iloc[test_index]
    

Feature Scaling

In [124]:
scale = StandardScaler()

In [125]:
X_train=scale.fit_transform(X_train)
X_test=scale.fit_transform(X_test)

Model Selection

In [203]:
log=LogisticRegression()
knn=KNeighborsClassifier(n_neighbors=5)

Fitting the model

In [204]:
log.fit(X_train,Y_train)
knn.fit(X_train,Y_train)

KNeighborsClassifier()

Testing the model

In [205]:
y_log_pred=log.predict(X_test)
y_knn_pred=knn.predict(X_test)

In [193]:
newdf=pd.DataFrame({'Actual':Y_test,'Log':y_log_pred,'KNN':y_knn_pred})

In [227]:
newdf.head()

,Actual,Log,KNN
266,0,0,0
267,0,0,0
269,0,0,0
270,0,1,1
276,0,0,0


Performance Analysis

In [206]:
log_mat=confusion_matrix(Y_test,y_log_pred)
knn_mat=confusion_matrix(Y_test,y_knn_pred)

In [207]:
print(f'Log Accuracy={np.ceil((log_mat[0][0]+log_mat[1][1])/log_mat.sum()*100)}%\t KNN Accuracy={np.ceil((knn_mat[0][0]+knn_mat[1][1])/knn_mat.sum()*100)}%\n')
print(f'Log Error Rate={np.ceil(((log_mat[0][1]+log_mat[1][0])/log_mat.sum())*100)}%\t KNN Error Rate={np.ceil(((knn_mat[0][1]+knn_mat[1][0])/knn_mat.sum())*100)}%\n')
print(f'Log Precision={np.ceil((log_mat[0][0])/(log_mat[0][0]+log_mat[1][0])*100)}%\t KNN Precision={np.ceil((knn_mat[0][0])/(knn_mat[0][0]+knn_mat[1][0])*100)}%\n')
print(f'Log Recall={np.ceil((log_mat[0][0])/(log_mat[0][0]+log_mat[0][1])*100)}%\t KNN Recall={np.ceil((knn_mat[0][0])/(knn_mat[0][0]+knn_mat[0][1])*100)}%\n')

Log Accuracy=73.0%	 KNN Accuracy=85.0%

Log Error Rate=28.0%	 KNN Error Rate=15.0%

Log Precision=74.0%	 KNN Precision=87.0%

Log Recall=91.0%	 KNN Recall=91.0%



HyperParameter Tuning

In [183]:
from sklearn.metrics import accuracy_score

In [223]:
k=[i for i in range(3,100,2)]

In [224]:
acc=[]
for i in k:
    hy_knn=KNeighborsClassifier(n_neighbors=i)
    hy_knn.fit(X_train,Y_train)
    Y_hy_knn_pred=hy_knn.predict(X_test)
    acc.append((np.round((accuracy_score(Y_test,Y_hy_knn_pred)*100),2),i))

print(sorted(acc,reverse=True))
    


[(88.75, 13), (88.75, 11), (87.5, 15), (86.25, 9), (85.0, 17), (85.0, 7), (85.0, 5), (83.75, 3), (80.0, 19), (78.75, 21), (77.5, 23), (76.25, 25), (73.75, 61), (73.75, 59), (73.75, 57), (73.75, 55), (73.75, 53), (73.75, 51), (73.75, 49), (73.75, 47), (73.75, 45), (73.75, 43), (73.75, 41), (73.75, 39), (73.75, 37), (73.75, 35), (73.75, 33), (73.75, 31), (73.75, 29), (73.75, 27), (72.5, 87), (72.5, 85), (72.5, 83), (72.5, 81), (72.5, 79), (72.5, 77), (72.5, 75), (72.5, 73), (72.5, 71), (72.5, 69), (72.5, 67), (72.5, 65), (72.5, 63), (71.25, 89), (70.0, 95), (70.0, 93), (70.0, 91), (68.75, 99), (68.75, 97)]


In [225]:
print('Max Accuracy =',sorted(acc)[-1][0],'%  at k =',sorted(acc)[-1][1]) 

Max Accuracy = 88.75 %  at k = 13
